In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time


In [2]:
# -----------------------------
# EDIT THESE
# -----------------------------
SEARCH_QUERY = "wireless headphones"   # change this
NUM_PAGES = 3                         # change this
BASE_URL = "https://www.amazon.com/s"
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# -----------------------------
# HELPER FUNCTIONS
# -----------------------------
def extract_rating(text):
    """
    Turns text like '4.5 out of 5 stars' into 4.5
    """
    if not text:
        return None
    match = re.search(r"\d+\.\d+", text)
    if match:
        return float(match.group())
    return None

# -----------------------------
# SCRAPE
# -----------------------------
rows = []

for page in range(1, NUM_PAGES + 1):
    params = {
        "k": SEARCH_QUERY,
        "page": page
    }

    response = requests.get(BASE_URL, params=params, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    # This matches the product cards in the search results.
    # If Amazon's structure changes, this is the first selector to adjust.
    product_cards = soup.select('div[role="listitem"]')

    for card in product_cards:
        # ---- TITLE ----
        # Based on your structure: h2 -> span
        title_tag = card.select_one("h2 span")
        title = title_tag.get_text(strip=True) if title_tag else None

        # ---- RATING ----
        # Based on your structure: data-cy="reviews block" -> span aria-hidden="true"
        rating_tag = card.select_one('div[data-cy="reviews block"] span[aria-hidden="true"]')

        # Fallback in case the above exact selector does not match
        if not rating_tag:
            rating_tag = card.select_one('span[aria-hidden="true"]')

        rating_text = rating_tag.get_text(strip=True) if rating_tag else None
        rating = extract_rating(rating_text)

        # Only keep rows where we at least have a title
        if title:
            rows.append({
                "title": title,
                "rating": rating,
                "search_query": SEARCH_QUERY,
                "page": page,
                "source": "Amazon"
            })

    time.sleep(2)

# -----------------------------
# DATASET
# -----------------------------
df = pd.DataFrame(rows)

# Remove duplicates if any
df = df.drop_duplicates()

print(df.head())
print(f"Collected {len(df)} rows")

Empty DataFrame
Columns: []
Index: []
Collected 0 rows
